In [ ]:
## 导入
import time
from pathlib import Path
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.b1500 import B1500

In [ ]:
## 基础配置
cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

# 当前已确认映射
CH_G = 4
CH_D = 5
CH_S = 6
CH_X = 7   # spare

# 默认量程与合规
VRANGE_G = 0
VRANGE_D = 0
VRANGE_S = 0
I_COMP_G = 1e-3
I_COMP_D = 1e-3
I_COMP_S = 1e-3

print("当前映射：")
print(f"G=CH{CH_G}, D=CH{CH_D}, S=CH{CH_S}, X=CH{CH_X}")

In [ ]:
## 扫描参数
# 小范围 Id-Vg 扫描参数
vg_points = [0.0, -0.2, -0.4, -0.6, -0.8, -1.0]
vd_fixed = 0.1
vs_fixed = 0.0
delay_s = 0.2

print("Vg points =", vg_points)
print("Vd fixed  =", vd_fixed)
print("Vs fixed  =", vs_fixed)
print("delay_s   =", delay_s)

In [ ]:
## 打开对话
session = VisaSession(cfg)
session.open()
b = B1500(session)

print("IDN:", session.query("*IDN?"))
print("ERRX start:", b.clear_err_queue())

In [ ]:
## 安全归零函数
def safe_zero_and_cl(channels):
    try:
        b.dz(channels)
    except Exception as e:
        print("DZ warning:", e)

    try:
        b.cl(channels)
    except Exception as e:
        print("CL warning:", e)

In [ ]:
## 单点测量函数
def measure_id_vg_point(vg, vd, vs, delay_s=0.2):
    row = {
        "vg_set": vg,
        "vd_set": vd,
        "vs_set": vs,
        "ig_raw": None,
        "id_raw": None,
        "is_raw": None,
        "ig_A": None,
        "id_A": None,
        "is_A": None,
        "err": None,
        "status": "ok",
    }

    try:
        # 推荐基础设置
        b.fmt(1)
        b.av(10, 1)
        b.fl(0)

        b.cn([CH_G, CH_D, CH_S])

        # 施加偏压
        b.dv(CH_G, VRANGE_G, vg, I_COMP_G)
        b.dv(CH_D, VRANGE_D, vd, I_COMP_D)
        b.dv(CH_S, VRANGE_S, vs, I_COMP_S)

        time.sleep(delay_s)

        # 读三端电流
        ig_raw = b._query(f"TI {CH_G},0", check_err=False)
        id_raw = b._query(f"TI {CH_D},0", check_err=False)
        is_raw = b._query(f"TI {CH_S},0", check_err=False)

        row["ig_raw"] = ig_raw
        row["id_raw"] = id_raw
        row["is_raw"] = is_raw

        row["ig_A"] = b._parse_scalar_response(ig_raw)
        row["id_A"] = b._parse_scalar_response(id_raw)
        row["is_A"] = b._parse_scalar_response(is_raw)

        row["err"] = b.errx()

    except Exception as e:
        row["status"] = "invalid"
        row["err"] = repr(e)

    finally:
        safe_zero_and_cl([CH_G, CH_D, CH_S])

    return row

In [ ]:
## 执行扫描
rows = []

for vg in vg_points:
    print(f"\n===== Id-Vg 点: Vg={vg} V, Vd={vd_fixed} V, Vs={vs_fixed} V =====")

    r = measure_id_vg_point(
        vg=vg,
        vd=vd_fixed,
        vs=vs_fixed,
        delay_s=delay_s,
    )
    rows.append(r)

    print("ig_raw =", r["ig_raw"])
    print("id_raw =", r["id_raw"])
    print("is_raw =", r["is_raw"])
    print("ig_A   =", r["ig_A"])
    print("id_A   =", r["id_A"])
    print("is_A   =", r["is_A"])
    print("err    =", r["err"])
    print("status =", r["status"])

df_idvg = pd.DataFrame(rows)

In [ ]:
## 查看结果、简单整理、保存，结束对话
df_idvg

df_idvg_sorted = df_idvg.sort_values("vg_set").reset_index(drop=True)
df_idvg_sorted[["vg_set", "id_A", "ig_A", "is_A", "status", "err"]]

run_dir = Path("runs") / time.strftime("%Y%m%d_%H%M%S_idvg_small_sweep")
run_dir.mkdir(parents=True, exist_ok=True)

df_idvg.to_csv(run_dir / "idvg_small_sweep.csv", index=False, encoding="utf-8-sig")
df_idvg.to_json(run_dir / "idvg_small_sweep.json", orient="records", force_ascii=False, indent=2)

print("结果保存到:", run_dir)

try:
    session.close()
    print("session 已关闭")
except Exception as e:
    print("close session error:", e)